In [ ]:
import torch
import torch.nn as nn
import torchvision
from torchvision import transforms
from tqdm import tqdm
from torch.utils.data import random_split, DataLoader

In [ ]:
# In case you want to do the computations on Cuda/mps instead of CPU

device = torch.device("cpu")
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.mps.is_available():
    device = torch.device("mps")

In [ ]:
# preprocessing pipeline to preprocess the images into the same preprocessing pipeline that
# the transfered base model (ResNet) has been trained on the feeded training images


preprocess = transforms.Compose(
    [
        transforms.Resize(512),
        # Data Augmentation to incrase the examples by adjusting original examples
        transforms.RandomRotation(10),
        transforms.RandomVerticalFlip(),
        transforms.RandomHorizontalFlip(),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.2),
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]
)

In [ ]:
# Full dataset:
# => https://www.kaggle.com/datasets/warcoder/tyre-quality-classification

# Collaborators: Chirag Chauhan

# License: CC BY 4.0
# => https://creativecommons.org/licenses/by/4.0/

# Attribution 4.0 International


dataset = torchvision.datasets.ImageFolder(
    root="path to image folder", transform=preprocess
)

In [ ]:
# seperating the data into training and validation datasets

train_dataset, val_dataset = random_split(dataset, [0.8, 0.2])

# packing the examples in each dataset to batch sizes of 32

train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=32, shuffle=True)

In [ ]:
# Loading the base model

resnet50_model = torchvision.models.resnet50(
    weights=torchvision.models.ResNet50_Weights.IMAGENET1K_V1
)

# clearing the last layer of the pretrained base model

resnet50_model.fc = nn.Identity()
# freezing the base model (excluding last classification layer) parameters and layers
for param in resnet50_model.parameters():
    param.requires_grad = False
resnet50_model.eval()
resnet50_model = resnet50_model.to(device)

In [ ]:
# defining the fully connected model that we want to add as last layers of the base model
fc_model = nn.Sequential(nn.Linear(2048, 1024), nn.ReLU(), nn.Linear(1024, 1))
fc_model = fc_model.to(device)

# attaching the fc model to the base model
model = nn.Sequential(resnet50_model, fc_model)
model = model.to(device)

In [ ]:
optimizer = torch.optim.Adam(fc_model.parameters(), lr=0.0002)
loss_fn = nn.BCEWithLogitsLoss()

for epoch in range(10):
    print(f"--- EPOCH: {epoch} ---")
    # puting the whole model on training mode
    model.train()
    # putting the ResNet part of the whole model (the base model part) on evaluation mode
    # so that its already trained parameters do not update
    resnet50_model.eval()

    loss_sum = 0
    train_accurate = 0
    train_sum = 0
    for X, Y in train_dataloader:
        # transfering the examples into desired device (gpu or cpu)
        X = X.to(device)
        Y = Y.to(device).type(torch.float).reshape(-1, 1)

        outputs = model(X)
        optimizer.zero_grad()
        loss = loss_fn(outputs, Y)
        loss_sum += loss.item()
        loss.backward()
        optimizer.step()

        predictions = torch.sigmoid(outputs) > 0.5
        accurate = (predictions == Y).sum().item()
        train_accurate += accurate
        train_sum += Y.size(0)
    # printing the accuracy of the training examples after each epoch, to see which
    # number of epoch has the best accuracy, and therefore using that specific
    # number of epoch saved model in next line of code
    print("Training loss: ", loss_sum / len(train_dataloader))
    print("Training accuracy: ", train_accurate / train_sum)

    # saving the model parameters for each epoch, so we have the option of using
    # the best model

    torch.save(fc_model.state_dict(), f"fc_model_{epoch}.pth")

In [ ]:
# Evaluating the model (the one after 10th epoch iterations) on validation dataset

model.eval()
    val_loss_sum = 0
    val_accurate = 0
    val_sum = 0
    with torch.no_grad():
        for X, Y in val_dataloader:
            X = X.to(device)
            Y = Y.to(device).type(torch.float).reshape(-1, 1)

            outputs = model(X)
            loss = loss_fn(outputs, Y)
            val_loss_sum+=loss.item()

            predictions = torch.sigmoid(outputs) > 0.5
            accurate = (predictions == Y).sum().item()
            val_accurate+=accurate
            val_sum+=Y.size(0)
    print("Validation loss: ", val_loss_sum / len(val_dataloader))
    print("Validation accuracy: ", val_accurate / val_sum)

# Now up to this point the training and validation phase has been completed.

in case we want to load the final model (the one with best training accuracy score which is the 3rd one) later and further evaluate it on custom images as test set to see the prediction (so now we just load the final model, so we don't have to train it from scratch each time), we use the following code:

In [ ]:
import torch
import torch.nn as nn
import torchvision
from torchvision import transforms
from torch.utils.data import random_split, DataLoader
from PIL import Image

In [ ]:
device = torch.device("cpu")
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.mps.is_available():
    device = torch.device("mps")

In [ ]:
# again preprocessing pipeline for custom images, but without the Data Augmentation part

preprocess = transforms.Compose(
    [
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]
)

In [ ]:
# same process as before, we load the base model, clear its last classification layer
# define the fc model that we want to attach to base model as last layers

resnet50_model = torchvision.models.resnet50(
    weights=torchvision.models.ResNet50_Weights.IMAGENET1K_V1
)
resnet50_model.fc = nn.Identity()
resnet50_model = resnet50_model.to(device)

fc_model = nn.Sequential(nn.Linear(2048, 1024), nn.ReLU(), nn.Linear(1024, 1))

# Now we just load the best fc model weights (in our experiment, it was the 3rd epoch model),
# and load the parameters on the defined fc model

fc_state_dict = torch.load("fc_model_3.pth", weights_only=True)
fc_model.load_state_dict(fc_state_dict)
fc_model = fc_model.to(device)

In [ ]:
# defining the whole model and putting it on evaluation mode

model = nn.Sequential(resnet50_model, fc_model)
model = model.to(device)
model.eval()

In [ ]:
# uploading the custom image (test set image) and preprocessing it

custom_image = Image.open("path_to_image.jpg")
custom_image_tensor = preprocess(custom_image)
custom_image_tensor = custom_image_tensor.unsqueeze(dim=0)
custom_image_tensor = custom_image_tensor.to(device)

In [ ]:
# inspecting the predicted probability

with torch.no_grad():
    y_pred = torch.sigmoid(model(tire_tensor))
    print(y_pred)